# Review segments and assigned topics

Run the cells below, then choose a **document index** or title. Switch between
**Both**, **Semantic**, and **Fixed**. Both shows the same talk side by side with each
model's native boundaries. Cards display full text, topic IDs and Estonian names,
timestamps, audio links and cluster membership strength.

Use **Unassigned only**, **Weak / unassigned**, or the search field to focus your review.
This notebook reads saved results; it does not fit models or call an API. Document
indices start at **0** and are sorted by title, with episode ID as a tie-breaker.

In [ ]:
import os
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
from IPython.display import HTML, clear_output, display

from arvamusfestivali_transcripts.topic_analysis.segment_review import (
    document_catalog,
    segment_review_html,
)

# Override TOPIC_REVIEW_DATASET to inspect another saved run.
project_root = Path(os.environ.get("TOPIC_ANALYSIS_PROJECT_ROOT", Path.cwd())).resolve()
if not (project_root / "data" / "topic-analysis").exists():
    if (project_root.parent / "data" / "topic-analysis").exists():
        project_root = project_root.parent
DATASET_PATH = Path(os.environ.get(
    "TOPIC_REVIEW_DATASET",
    str(project_root / "data/topic-analysis/results/segment-dataset/all-segments.parquet"),
))
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Saved segment dataset missing: {DATASET_PATH}")
segments = pd.read_parquet(DATASET_PATH)
segments["episode_id"] = segments.episode_id.astype(str)
documents = document_catalog(segments)
if documents.empty:
    raise ValueError("The dataset contains no documents")
print(f"Loaded {len(segments):,} segments from {len(documents)} documents.")
print(DATASET_PATH)

In [ ]:
# Catalog: these indices select the same recording in both modes.
display(documents.rename_axis("document_index"))

In [ ]:
document_index = widgets.BoundedIntText(
    value=0, min=0, max=len(documents) - 1, description="Doc index:",
)
document_title = widgets.Dropdown(
    options=[(f"{i}: {row.talk_name}", i) for i, row in documents.iterrows()],
    value=0, description="Document:", layout=widgets.Layout(width="95%"),
)
index_link = widgets.link((document_index, "value"), (document_title, "value"))
previous = widgets.Button(description="← Previous")
following = widgets.Button(description="Next →")
previous.on_click(lambda _: setattr(document_index, "value", max(0, document_index.value - 1)))
following.on_click(lambda _: setattr(
    document_index, "value", min(len(documents) - 1, document_index.value + 1)
))
mode = widgets.ToggleButtons(
    options=[("Both", "both"), ("Semantic", "semantic"), ("Fixed", "fixed")],
    value="both", description="Mode:",
)
unassigned = widgets.Checkbox(value=False, description="Unassigned only")
weak = widgets.Checkbox(value=False, description="Weak / unassigned")
threshold = widgets.FloatSlider(
    value=0.5, min=0, max=1, step=0.05, description="Weak ≤", continuous_update=False,
)
search = widgets.Text(
    placeholder="Search segment text or topic names", description="Search:",
    continuous_update=False, layout=widgets.Layout(width="95%"),
)
review_output = widgets.Output()


def show_review(change=None):
    with review_output:
        clear_output(wait=True)
        display(HTML(segment_review_html(
            segments, document_index.value, mode=mode.value,
            only_unassigned=unassigned.value, weak_only=weak.value,
            weak_threshold=threshold.value, search=search.value,
        )))


for control in (document_index, mode, unassigned, weak, threshold, search):
    control.observe(show_review, names="value")
display(widgets.VBox([
    document_title,
    widgets.HBox([previous, document_index, following]),
    mode,
    widgets.HBox([unassigned, weak]),
    threshold,
    search,
    review_output,
]))
show_review()

## Direct selection without widgets

The cell below also works in notebook viewers without interactive widget support.
Change `DOC_INDEX` and `MODE`, then rerun it. This view has independent settings.

In [ ]:
DOC_INDEX = 0
MODE = "both"  # "both", "semantic", or "fixed"
# Uncomment to display a second, non-widget review:
# display(HTML(segment_review_html(segments, DOC_INDEX, mode=MODE)))